# Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf
from pmdarima import auto_arima
import pandas as pd
import matplotlib.pyplot as plt

# Load Data

In [ ]:
data_path = "datasets"

In [ ]:
ts = pd.read_csv(f"{data_path}/dataset.csv", header=0, parse_dates=True, index_col=0)

In [ ]:
ts.index

In [ ]:
ts.columns

In [ ]:
ts.info()

In [ ]:
ts.head()

In [ ]:
plt.plot(ts[:7])
plt.plot(ts[7:14])
plt.plot(ts[14:21])
plt.plot(ts[21:28])
plt.show()

In [ ]:
plt.plot(ts)
plt.show()

In [ ]:
plt.plot(ts.diff())
plt.show()

# Time Series Decomposition

In [ ]:
# DecomposeResult
# A object with seasonal, trend, and resid attributes.

decomp_ts = seasonal_decompose(ts, model="add", period=7)
decomp_ts

In [ ]:
decomp_ts.trend.to_csv(f"{data_path}/decomp/trend.csv")
decomp_ts.seasonal.to_csv(f"{data_path}/decomp/seasonal.csv")
decomp_ts.resid.to_csv(f"{data_path}/decomp/resid.csv")

In [ ]:
plt.plot(ts, c='b', label=f"Original Time Series")
plt.plot(decomp_ts.trend, c='g', label=f"Decomposition: Trend")
plt.legend()
plt.show()

In [ ]:
# Weak seasonality

plt.plot(decomp_ts.seasonal)
plt.show()

In [ ]:
plt.plot(decomp_ts.resid)
plt.show()

# Auto Arima

In [ ]:
auto_arima_model = auto_arima(y = ts, seasonal=False, trace=True)

In [ ]:
auto_arima_model.plot_diagnostics()

In [ ]:
best_order = auto_arima_model.order
print(f"Best order: {best_order}")

# Fit Arima model with best order

In [ ]:
arima_model = ARIMA(endog=ts, order=best_order, freq='D')
arima_model

In [ ]:
model_res = arima_model.fit()

In [ ]:
# ARIMAResults
# -> fittedvalues: in_sample predictions
# -> resid: errors of the predictions
# resid = ground_truth - fittedvalues

(model_res.fittedvalues + model_res.resid) == ts['values']

In [ ]:
plt.plot(ts[1:], label=f"Original Time Series")
plt.plot(model_res.fittedvalues[1:], label=f"Fitted values")
plt.legend()
plt.show()

In [ ]:
plt.plot(model_res.resid[1:], label=f"Residuals")
plt.legend()
plt.show()

# Forecast

In [ ]:
forecast_df = model_res.forecast(steps=10)
forecast_df.name = 'forecast'

In [ ]:
forecast_df

In [ ]:
forecast_df.to_frame()

In [ ]:
forecast_df.to_frame().to_csv(f"{data_path}/forecast.csv")

In [ ]:
plt.plot(ts[-30:])
plt.plot(forecast_df)
plt.show()

# Stationary Test

In [ ]:
if adfuller(model_res.resid)[1] < 0.05:
    print(f"Model is stationary.")

In [ ]:
acf_values = acf(model_res.resid)
plt.plot(acf_values[1:])
plt.show()

# End